# Module 2 — Fama-MacBeth Regression

## Overview

This module answers the central question of cross-sectional asset
pricing: **which of the 41 characteristics are priced?** A characteristic
is priced if there is a statistically significant relationship between
a stock's exposure to that characteristic and its expected return,
meaning investors require compensation for bearing that exposure.

We use the **Fama-MacBeth (1973)** procedure, the standard method
in empirical asset pricing for estimating factor risk prices from
panel data.

## The Fama-MacBeth Procedure

**Pass 1 — Factor loadings**

In the original Fama-MacBeth (1973) paper, Pass 1 runs a time-series
regression for each stock to estimate its sensitivity (beta) to each
factor. In our framework we skip this step. Following Fama \& French
(1992) and CPZ (2020), we use the rank-normalized characteristics
$\tilde{z}_{i,t,j}$ directly as factor loadings. This is justified
because characteristics and covariance-based betas span the same
space, that is, a stock's book-to-market rank captures its value exposure
as precisely as a time-series regression would.

**Pass 2 — Monthly cross-sectional regressions**

Each month $t$, we regress the cross-section of stock excess returns
on the characteristics:

$$r_{i,t}^e = \sum_{j=1}^{J} \tilde{z}_{i,t,j}\,\lambda_{j,t}
  + \eta_{i,t}, \qquad i = 1, \ldots, N_t$$

This yields a monthly risk price estimate $\hat{\lambda}_{j,t}$ for
each characteristic $j$. Running $T$ such regressions gives the full
time series $\{\hat{\lambda}_{j,t}\}_{t=1}^{T}$.

**FM estimate and inference**

The Fama-MacBeth risk price is the time-series average:

$$\hat{\lambda}_j^{\text{FM}} = \frac{1}{T}
  \sum_{t=1}^{T} \hat{\lambda}_{j,t}$$

We test $H_0: \lambda_j = 0$ using **Newey-West HAC standard errors**
with lag $L = \lfloor T^{1/4} \rfloor$, which corrects for serial
correlation in $\hat{\lambda}_{j,t}$ caused by persistent economic
conditions. We do not apply the Shanken (1992) EIV correction because
(i) we use directly observable characteristics rather than estimated
betas, so there is no first-pass estimation error, and (ii) with
$p = 41$ factors the correction breaks down numerically.

## The Multicollinearity Problem and Our Solution

With 41 correlated characteristics as joint regressors, the condition
number of $X'X$ is 8,279 and some VIFs exceed 400. Joint OLS Pass 2
is numerically unstable -- individual $\hat{\lambda}_{j,t}$ become
implausibly large as correlated regressors absorb each other's effects.

Following Koijen \& Van Nieuwerburgh (2020) and standard practice in
high-dimensional asset pricing, we implement three complementary
approaches:

| Approach | Method | Purpose |
|----------|--------|---------|
| **Univariate FM** | 41 separate single-characteristic regressions | Primary result -- no multicollinearity |
| **Ridge FM** | Joint regression with L2 penalty $\alpha=0.01$ | Robustness -- stabilized joint estimates |
| **LASSO FM** | Joint regression with L1 penalty | Variable selection -- which factors survive competition |

The **univariate FM is our headline result**. A factor is priced if
its univariate $|t_{\text{NW}}| > 1.96$.

## Inputs and Outputs

**Input:** `outputs/M1/factor_returns_41.parquet` and the raw panel
from the CPZ extended dataset.

**Outputs** (saved to `outputs/M2/`):
- `lambda_univariate.csv` — univariate FM risk prices and $t$-statistics
- `lambda_ridge.csv` — ridge FM results
- `lambda_lasso.csv` — LASSO FM results
- `lambda_timeseries_univariate.parquet` — monthly $\hat{\lambda}_{j,t}$ (feeds M4)
- `r2_timeseries.csv` — monthly cross-sectional $R^2$ (ridge)
- `figures/` — risk price bar chart, $R^2$ time series

In [9]:
# Imports and Configuration

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.linear_model import Ridge, Lasso
import os, warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook"

# ── Paths ─────────────────────────────────────────────────────
PROJECT_ROOT = (r"C:\Users\amosa\Documents\My Graduate School"
                r"\SPRING 2026\Courses\AMS520_Machine Learning in Finance"
                r"\Project\General ML4Trading Resource"
                r"\Personal_Fundamental Factor Investing"
                r"\fundamental-factor-investing")
DATA_DIR = r"C:\Users\amosa\ml4t_data\extended_v2"
M1_OUT   = os.path.join(PROJECT_ROOT, "outputs", "M1")
OUT_DIR  = os.path.join(PROJECT_ROOT, "outputs", "M2")
FIG_DIR  = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── 41 characteristics by economic family ─────────────────────
CHAR_FAMILIES = {
    "Value":         ["BEME","E2P","CF2P","D2P","S2P","A2ME"],
    "Profitability": ["PROF","ROE","ROA","OP","PM","PCM","RNA"],
    "Investment":    ["Investment","NOA","DPI2A","NI","OA","AC"],
    "Momentum":      ["r12_2","r2_1","r12_7","r36_13",
                      "LT_Rev","SUV","Rel2High"],
    "Risk":          ["Beta","IdioVol","Variance",
                      "Spread","LTurnover","LME"],
    "Other":         ["Q","C","AT","ATO","CTO",
                      "D2A","FC2Y","Lev","SGA2S"],
}
ALL_41 = [c for fam in CHAR_FAMILIES.values() for c in fam]

FAMILY_COLORS = {
    "Value":         "#1f77b4",
    "Profitability": "#2ca02c",
    "Investment":    "#ff7f0e",
    "Momentum":      "#d62728",
    "Risk":          "#9467bd",
    "Other":         "#8c564b",
}

print(f"Output directory: {OUT_DIR}")
print(f"Factors:          {len(ALL_41)}")

Output directory: C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS520_Machine Learning in Finance\Project\General ML4Trading Resource\Personal_Fundamental Factor Investing\fundamental-factor-investing\outputs\M2
Factors:          41


In [10]:
# Load data

print("Loading data...")

# Factor returns from M1 (696 × 41)
factor_returns = pd.read_parquet(
    os.path.join(M1_OUT, "factor_returns_41.parquet"))
factor_returns.index = (pd.to_datetime(factor_returns.index)
                        + pd.offsets.MonthEnd(0))
factor_returns = factor_returns[ALL_41]

# Raw panel: characteristics + stock returns
train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
valid = pd.read_parquet(os.path.join(DATA_DIR, "valid.parquet"))
test  = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

panel = pd.concat([train, valid, test], ignore_index=True)
panel['date'] = pd.to_datetime(panel['date']) + pd.offsets.MonthEnd(0)
panel = panel.sort_values(['date','permno']).reset_index(drop=True)

# Newey-West lag: L = floor(T^0.25), standard for monthly data
T = len(factor_returns)
L = int(np.floor(T ** 0.25))

print(f"Factor returns:  {factor_returns.shape}  "
      f"({factor_returns.index.min().date()} — "
      f"{factor_returns.index.max().date()})")
print(f"Panel:           {panel.shape[0]:,} stock-month observations")
print(f"Unique months:   {panel['date'].nunique()}")
print(f"T = {T},  Newey-West lag L = floor({T}^0.25) = {L}")

Loading data...
Factor returns:  (696, 41)  (1967-01-31 — 2024-12-31)
Panel:           1,605,189 stock-month observations
Unique months:   696
T = 696,  Newey-West lag L = floor(696^0.25) = 5


----

## Newey-West Standard Errors

After collecting the $T$ monthly estimates
$\hat{\lambda}_{j,1}, \ldots, \hat{\lambda}_{j,T}$, we test
$H_0: \lambda_j = 0$ using the $t$-statistic:

$$t_j = \frac{\hat{\lambda}_j^{\text{FM}}}{\widehat{\text{SE}}_j^{\text{NW}}}$$

The **raw FM standard error** assumes the monthly estimates are
independent:

$$\widehat{\text{SE}}_j^{\text{raw}} = \frac{s(\hat{\lambda}_{j,t})}{\sqrt{T}}$$

The **Newey-West HAC standard error** corrects for serial correlation
using the Bartlett kernel with $L = \lfloor T^{1/4} \rfloor$ lags:

$$\hat{V}_j^{\text{NW}} = \hat{\gamma}_0
  + 2\sum_{\ell=1}^{L}\!\left(1 - \frac{\ell}{L+1}\right)\hat{\gamma}_\ell$$

where $\hat{\gamma}_\ell$ is the sample autocovariance of
$\hat{\lambda}_{j,t}$ at lag $\ell$.

The Newey-West SE is always $\geq$ the raw SE — it is the more
conservative and correct estimate for our setting. We use
$|t_{\text{NW}}| > 1.96$ as our significance criterion.

In [11]:
# ── Newey-West HAC standard error ─────────────────────────────
def newey_west_se(series, lags):
    """
    Newey-West HAC standard error for the mean of a time series.
    Bartlett kernel, lag truncation L = floor(T^0.25).
    Drops NaN before computing.
    Returns NaN if fewer than lags+2 observations.
    """
    x  = series.dropna().values
    T_ = len(x)
    if T_ < lags + 2:
        return np.nan
    dev    = x - x.mean()
    nw_var = (dev ** 2).mean()
    for ell in range(1, lags + 1):
        w       = 1.0 - ell / (lags + 1)   # Bartlett weight
        nw_var += 2 * w * (dev[ell:] * dev[:-ell]).mean()
    return np.sqrt(max(nw_var, 0.0) / T_)

In [12]:
# ── Approach 1: Univariate Fama-MacBeth ───────────────────────
# For each of the 41 characteristics independently, run T monthly
# cross-sectional regressions:
#   r_{i,t}^e = alpha_t + lambda_{j,t} * z̃_{i,t,j} + η_{i,t}
# No multicollinearity by construction.

print("=== Univariate Fama-MacBeth ===")
print(f"41 characteristics × {T} months = {41*T:,} regressions\n")

dates = sorted(panel['date'].unique())

lambda_uni_records = []   # one row per month, one col per char
r2_uni_records     = []   # same structure for R²

for date in dates:
    month_df = panel[panel['date'] == date]
    y        = month_df['ret_excess'].values
    row_lam  = {'date': date}
    row_r2   = {'date': date}

    for char in ALL_41:
        if char not in month_df.columns:
            row_lam[char] = np.nan
            row_r2[char]  = np.nan
            continue

        x     = month_df[char].values
        valid = ~np.isnan(x)
        if valid.sum() < 50:
            row_lam[char] = np.nan
            row_r2[char]  = np.nan
            continue

        y_v = y[valid]
        # Include intercept so λ measures the slope on z̃ alone
        X_v = np.column_stack([np.ones(valid.sum()), x[valid]])

        try:
            coef  = np.linalg.lstsq(X_v, y_v, rcond=None)[0]
            lam   = coef[1]                  # slope on characteristic
            resid = y_v - X_v @ coef
            ss_res = resid @ resid
            ss_tot = ((y_v - y_v.mean()) ** 2).sum()
            r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        except Exception:
            lam, r2 = np.nan, np.nan

        row_lam[char] = lam
        row_r2[char]  = r2

    lambda_uni_records.append(row_lam)
    r2_uni_records.append(row_r2)

# Assemble into DataFrames
lambda_uni_ts = pd.DataFrame(lambda_uni_records).set_index('date')
lambda_uni_ts.index = pd.to_datetime(lambda_uni_ts.index) + pd.offsets.MonthEnd(0)

r2_uni_ts = pd.DataFrame(r2_uni_records).set_index('date')
r2_uni_ts.index = lambda_uni_ts.index

# ── FM point estimates and standard errors ────────────────────
results_uni = []
for char in ALL_41:
    s     = lambda_uni_ts[char]
    lam   = s.mean()
    n_obs = int(s.notna().sum())
    se_r  = s.std() / np.sqrt(n_obs)        # raw FM SE
    se_n  = newey_west_se(s, L)             # Newey-West SE
    t_raw = lam / se_r if se_r > 0 else np.nan
    t_nw  = lam / se_n if (se_n and se_n > 0) else np.nan

    results_uni.append({
        'Characteristic': char,
        'Family':         next(f for f, cs in CHAR_FAMILIES.items()
                               if char in cs),
        'Lambda (ann %)': round(lam * 12 * 100, 3),
        'SE (NW)':        se_n,
        't (raw)':        round(t_raw, 3) if not np.isnan(t_raw) else np.nan,
        't (NW)':         round(t_nw,  3) if not np.isnan(t_nw)  else np.nan,
        'N months':       n_obs,
    })

fm_uni = pd.DataFrame(results_uni).set_index('Characteristic')

n_196 = (fm_uni['t (NW)'].abs() > 1.96).sum()
n_300 = (fm_uni['t (NW)'].abs() > 3.00).sum()
print(f"Univariate FM complete.")
print(f"Factors priced |t_NW| > 1.96: {n_196}/41")
print(f"Factors priced |t_NW| > 3.00: {n_300}/41")

=== Univariate Fama-MacBeth ===
41 characteristics × 696 months = 28,536 regressions

Univariate FM complete.
Factors priced |t_NW| > 1.96: 28/41
Factors priced |t_NW| > 3.00: 24/41


In [13]:
# ── Approach 2: Ridge Fama-MacBeth ────────────────────────────
# Joint regression each month with L2 penalty α = 0.01.
# Ridge shrinks correlated coefficients toward zero, stabilising
# estimates without eliminating any factor entirely.
# α = 0.01 is small relative to the scale of X'X (~2,300 stocks)
# so it regularises without overwhelming the signal.

print("=== Ridge Fama-MacBeth (α = 0.01) ===")
print(f"Joint ridge regression × {T} months\n")

RIDGE_ALPHA = 0.01

lambda_ridge_records = []
r2_ridge_list        = []

for date in dates:
    month_df = panel[panel['date'] == date]
    chars_ok = [c for c in ALL_41
                if c in month_df.columns
                and month_df[c].notna().all()]
    if len(chars_ok) < 10 or len(month_df) < 50:
        continue

    y = month_df['ret_excess'].values
    X = month_df[chars_ok].values

    try:
        mdl   = Ridge(alpha=RIDGE_ALPHA, fit_intercept=False)
        mdl.fit(X, y)
        y_hat = mdl.predict(X)
        resid = y - y_hat
        ss_res = resid @ resid
        ss_tot = ((y - y.mean()) ** 2).sum()
        r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    except Exception:
        continue

    row = {'date': date, 'R2': r2}
    for j, char in enumerate(chars_ok):
        row[char] = mdl.coef_[j]
    lambda_ridge_records.append(row)
    r2_ridge_list.append({'date': date, 'R2': r2})

lambda_ridge_ts = pd.DataFrame(lambda_ridge_records).set_index('date')
lambda_ridge_ts.index = (pd.to_datetime(lambda_ridge_ts.index)
                         + pd.offsets.MonthEnd(0))
r2_ridge = lambda_ridge_ts.pop('R2')
lambda_ridge_ts = lambda_ridge_ts.reindex(columns=ALL_41)

results_ridge = []
for char in ALL_41:
    s     = lambda_ridge_ts[char]
    lam   = s.mean()
    n_obs = int(s.notna().sum())
    se_r  = s.std() / np.sqrt(n_obs)
    se_n  = newey_west_se(s, L)
    t_raw = lam / se_r if se_r > 0 else np.nan
    t_nw  = lam / se_n if (se_n and se_n > 0) else np.nan

    results_ridge.append({
        'Characteristic': char,
        'Family':         next(f for f, cs in CHAR_FAMILIES.items()
                               if char in cs),
        'Lambda (ann %)': round(lam * 12 * 100, 3),
        'SE (NW)':        se_n,
        't (raw)':        round(t_raw, 3) if not np.isnan(t_raw) else np.nan,
        't (NW)':         round(t_nw,  3) if not np.isnan(t_nw)  else np.nan,
        'N months':       n_obs,
    })

fm_ridge = pd.DataFrame(results_ridge).set_index('Characteristic')

n_196r = (fm_ridge['t (NW)'].abs() > 1.96).sum()
print(f"Ridge FM complete.")
print(f"Mean monthly R²: {r2_ridge.mean()*100:.2f}%")
print(f"Factors priced |t_NW| > 1.96: {n_196r}/41")

=== Ridge Fama-MacBeth (α = 0.01) ===
Joint ridge regression × 696 months

Ridge FM complete.
Mean monthly R²: 24.44%
Factors priced |t_NW| > 1.96: 35/41


In [14]:
# ── Approach 3: LASSO Fama-MacBeth ────────────────────────────
# Joint regression with L1 penalty α = 0.0005.
# LASSO zeros out redundant characteristics each month,
# performing automatic variable selection.
# α chosen to keep ~20 active factors on average — small enough
# to preserve economically important signals, large enough to
# eliminate near-duplicates.

print("=== LASSO Fama-MacBeth (α = 0.0005) ===")
print(f"Joint LASSO regression × {T} months\n")

LASSO_ALPHA = 0.0005

lambda_lasso_records = []
r2_lasso_list        = []
n_active_per_month   = []

for date in dates:
    month_df = panel[panel['date'] == date]
    chars_ok = [c for c in ALL_41
                if c in month_df.columns
                and month_df[c].notna().all()]
    if len(chars_ok) < 10 or len(month_df) < 50:
        continue

    y = month_df['ret_excess'].values
    X = month_df[chars_ok].values

    try:
        mdl = Lasso(alpha=LASSO_ALPHA, fit_intercept=False,
                    max_iter=5000, tol=1e-4)
        mdl.fit(X, y)
        y_hat = mdl.predict(X)
        resid = y - y_hat
        ss_res = resid @ resid
        ss_tot = ((y - y.mean()) ** 2).sum()
        r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        n_active_per_month.append((mdl.coef_ != 0).sum())
    except Exception:
        continue

    row = {'date': date, 'R2': r2}
    for j, char in enumerate(chars_ok):
        row[char] = mdl.coef_[j]
    lambda_lasso_records.append(row)
    r2_lasso_list.append({'date': date, 'R2': r2})

lambda_lasso_ts = pd.DataFrame(lambda_lasso_records).set_index('date')
lambda_lasso_ts.index = (pd.to_datetime(lambda_lasso_ts.index)
                         + pd.offsets.MonthEnd(0))
r2_lasso = lambda_lasso_ts.pop('R2')
lambda_lasso_ts = lambda_lasso_ts.reindex(columns=ALL_41, fill_value=0.0)

results_lasso = []
for char in ALL_41:
    s          = lambda_lasso_ts[char]
    lam        = s.mean()
    n_obs      = int(s.notna().sum())
    pct_active = (s != 0).mean() * 100
    std_       = s.std()
    se_r  = std_ / np.sqrt(n_obs) if std_ > 0 else np.nan
    se_n  = newey_west_se(s, L)
    t_raw = lam / se_r if (se_r and se_r > 0) else np.nan
    t_nw  = lam / se_n if (se_n and se_n > 0) else np.nan

    results_lasso.append({
        'Characteristic': char,
        'Family':         next(f for f, cs in CHAR_FAMILIES.items()
                               if char in cs),
        'Lambda (ann %)': round(lam * 12 * 100, 3),
        'SE (NW)':        se_n,
        't (raw)':        round(t_raw, 3) if (t_raw and not np.isnan(t_raw)) else np.nan,
        't (NW)':         round(t_nw,  3) if (t_nw  and not np.isnan(t_nw))  else np.nan,
        '% Active':       round(pct_active, 1),
        'N months':       n_obs,
    })

fm_lasso = pd.DataFrame(results_lasso).set_index('Characteristic')

n_active = (fm_lasso['% Active'] > 50).sum()
n_196l   = (fm_lasso['t (NW)'].abs() > 1.96).sum()
print(f"LASSO FM complete.")
print(f"Mean monthly R²:              {r2_lasso.mean()*100:.2f}%")
print(f"Mean active factors/month:    {np.mean(n_active_per_month):.1f}/41")
print(f"Factors active >50% months:   {n_active}/41")
print(f"Factors priced |t_NW| > 1.96: {n_196l}/41")

=== LASSO Fama-MacBeth (α = 0.0005) ===
Joint LASSO regression × 696 months

LASSO FM complete.
Mean monthly R²:              21.14%
Mean active factors/month:    21.0/41
Factors active >50% months:   17/41
Factors priced |t_NW| > 1.96: 32/41


----

## Results: Risk Price Estimates

The table below reports the annualized FM risk price
$\hat{\lambda}_j^{\text{FM}} \times 12$ for each of the 41
characteristics under all three approaches, with Newey-West
$t$-statistics.

**Reading the signs.** In the univariate regression, a negative
$\hat{\lambda}_j$ means high $\tilde{z}_{i,t,j}$ is associated
with low returns. For value factors (BEME, E2P, etc.) in the CPZ
dataset, a high $\tilde{z}$ corresponds to a *growth* stock — so a
negative coefficient is economically correct (growth stocks earn
lower returns than value stocks). The IC-based flip in M1 corrected
the portfolio construction direction but does not affect the
cross-sectional regression, which uses the raw characteristic.

**Significance thresholds:**
- `*` — $|t_{\text{NW}}| > 1.96$ (5% level)
- `**` — $|t_{\text{NW}}| > 3.00$ (Harvey, Liu \& Zhu 2016 threshold)

**Priced** = $|t_{\text{NW}}| > 1.96$ in the univariate column.

**% Active** (LASSO only) = fraction of months the characteristic
received a nonzero coefficient. Consistently active factors are
robust to competition from all 40 others.

In [15]:
# Results table

print(f"{'─'*92}")
print(f"{'Characteristic':<16} {'Family':<15} "
      f"{'Uni λ%':>8} {'t_NW':>7}  "
      f"{'Rdg λ%':>8} {'t_NW':>7}  "
      f"{'LSO λ%':>8} {'t_NW':>7}  {'%Act':>5}  Priced?")
print(f"{'─'*92}")

n_uni = n_rdg = n_lso = 0

for fam in CHAR_FAMILIES:
    print(f"\n  ── {fam} ──")
    for char in CHAR_FAMILIES[fam]:
        u  = fm_uni.loc[char]
        r  = fm_ridge.loc[char]
        l  = fm_lasso.loc[char]
        tu = u['t (NW)']
        tr = r['t (NW)']
        tl = l['t (NW)']

        def sig(t):
            if pd.isna(t): return "  "
            return "**" if abs(t) > 3 else ("*" if abs(t) > 1.96 else "  ")

        priced = "✓" if (not pd.isna(tu) and abs(tu) > 1.96) else " "
        if not pd.isna(tu) and abs(tu) > 1.96: n_uni += 1
        if not pd.isna(tr) and abs(tr) > 1.96: n_rdg += 1
        if not pd.isna(tl) and abs(tl) > 1.96: n_lso += 1

        tl_str = f"{tl:>6.2f}" if not pd.isna(tl) else "   n/a"
        tr_str = f"{tr:>6.2f}" if not pd.isna(tr) else "   n/a"

        print(f"  {char:<16} {u['Family']:<15} "
              f"{u['Lambda (ann %)']:>8.2f} {tu:>6.2f}{sig(tu)}  "
              f"{r['Lambda (ann %)']:>8.2f} {tr_str}{sig(tr)}  "
              f"{l['Lambda (ann %)']:>8.2f} {tl_str}{sig(tl)}  "
              f"{l['% Active']:>4.0f}%  {priced}")

print(f"\n{'─'*92}")
print(f"* |t|>1.96   ** |t|>3.00   Priced = univariate |t_NW| > 1.96")
print(f"\nFactors priced (|t_NW| > 1.96):")
print(f"  Univariate:  {n_uni}/41  ← primary result")
print(f"  Ridge:       {n_rdg}/41")
print(f"  LASSO:       {n_lso}/41")

────────────────────────────────────────────────────────────────────────────────────────────
Characteristic   Family            Uni λ%    t_NW    Rdg λ%    t_NW    LSO λ%    t_NW   %Act  Priced?
────────────────────────────────────────────────────────────────────────────────────────────

  ── Value ──
  BEME             Value             -65.88 -19.59**   -136.15 -18.32**    -37.53 -12.89**    75%  ✓
  E2P              Value             -29.61  -8.82**      4.20   1.38        0.07   0.06      40%  ✓
  CF2P             Value             -37.31 -11.15**     14.10   7.08**      1.08   0.92      44%  ✓
  D2P              Value             -13.54  -4.32**     10.56   8.79**      2.56   3.76**    42%  ✓
  S2P              Value             -41.11 -11.90**    -63.21 -15.01**    -13.97  -7.43**    36%  ✓
  A2ME             Value             -59.66 -16.74**   -266.73 -20.49**    -56.46  -8.65**    66%  ✓

  ── Profitability ──
  PROF             Profitability       5.96   3.36**    -16.31  -6.6

----

## Cross-Sectional $R^2$

The monthly $R^2$ measures what fraction of the cross-sectional
variation in stock returns is explained by the characteristics
in that month. The remainder is firm-specific noise — earnings
surprises, analyst revisions, idiosyncratic events — that no
characteristic-based model can predict in advance.

A few benchmarks from the literature:

| Study | Characteristics | Mean Cross-Sectional $R^2$ |
|-------|----------------|--------------------------|
| Fama \& French (1992) | 5 | ~3–5% |
| Freyberger et al. (2020) | 62 | ~10–15% |
| CPZ (2020) | 46 | ~15–20% |
| **This project (Ridge)** | **41** | **~24%** |

The joint ridge model achieves 24% — above CPZ's benchmark,
consistent with our longer sample and rank-normalized inputs.

**Negative $R^2$** occurs in extreme crash months (e.g. October 1987,
Black Monday) when all stocks move together and cross-sectional
variation collapses. These observations are genuine and are kept
in the sample -- the Newey-West correction absorbs their effect
on inference.

In [18]:
# R^2 analysis and plot

print("=== Cross-sectional R² analysis ===\n")

r2_uni_mean = r2_uni_ts[ALL_41].mean(axis=1).dropna()

print(f"Univariate mean R² (avg over 41 factors):")
print(f"  Mean:   {r2_uni_mean.mean()*100:.2f}%")
print(f"  Median: {r2_uni_mean.median()*100:.2f}%")

print(f"\nRidge joint R²:")
print(f"  Mean:   {r2_ridge.mean()*100:.2f}%")
print(f"  Median: {r2_ridge.median()*100:.2f}%")
print(f"  Min:    {r2_ridge.min():.4f}  ({r2_ridge.idxmin().strftime('%b %Y')})")
print(f"  Max:    {r2_ridge.max():.4f}  ({r2_ridge.idxmax().strftime('%b %Y')})")

print(f"\nLASSO joint R²:")
print(f"  Mean:   {r2_lasso.mean()*100:.2f}%")
print(f"  Median: {r2_lasso.median()*100:.2f}%")

print(f"\nRidge R² by decade:")
for start in range(1967, 2025, 10):
    end  = min(start + 9, 2024)
    mask = (r2_ridge.index.year >= start) & (r2_ridge.index.year <= end)
    sub  = r2_ridge[mask]
    if len(sub):
        print(f"  {start}s: mean={sub.mean():.3f}  "
              f"std={sub.std():.3f}  n={len(sub)}")

# ── Plot ridge R² time series ─────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=r2_ridge.index, y=r2_ridge.values * 100,
    name='Monthly R² (Ridge)',
    mode='lines',
    line=dict(color='steelblue', width=0.8),
    opacity=0.45,
    hovertemplate="Date: %{x|%b %Y}<br>R²: %{y:.2f}%<extra></extra>"
))

r2_ann = r2_ridge.resample('YE').mean()
fig.add_trace(go.Scatter(
    x=r2_ann.index, y=r2_ann.values * 100,
    name='Annual average',
    mode='lines+markers',
    line=dict(color='darkblue', width=2.5),
    marker=dict(size=5),
    hovertemplate="Year: %{x|%Y}<br>R²: %{y:.2f}%<extra></extra>"
))

fig.add_hline(
    y=r2_ridge.mean() * 100,
    line=dict(color='crimson', width=1.5, dash='dash'),
    annotation_text=f"Mean = {r2_ridge.mean()*100:.1f}%",
    annotation_position="right"
)

fig.update_layout(
    title=dict(
        text=("<b>Cross-Sectional R² Over Time — Ridge Joint Model</b><br>"
              "<sup>Monthly and annual average | 41 characteristics | "
              "1967–2024</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Date', showgrid=True,
               gridcolor='rgba(200,200,200,0.3)'),
    yaxis=dict(title='Cross-Sectional R² (%)', ticksuffix='%',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    legend=dict(x=3.31, y=0.5),
    width=1000, height=420,
    margin=dict(l=60, r=130, t=80, b=60),
    hovermode='x'
)
fig.show()
fig.write_image(os.path.join(FIG_DIR, "r2_timeseries.png"), scale=2)
print("Saved: r2_timeseries.png")

=== Cross-sectional R² analysis ===

Univariate mean R² (avg over 41 factors):
  Mean:   1.58%
  Median: 1.22%

Ridge joint R²:
  Mean:   24.44%
  Median: 31.44%
  Min:    -4.1338  (Oct 1987)
  Max:    0.6478  (Jan 1968)

LASSO joint R²:
  Mean:   21.14%
  Median: 28.17%

Ridge R² by decade:
  1967s: mean=0.148  std=0.465  n=120
  1977s: mean=0.221  std=0.385  n=120
  1987s: mean=0.261  std=0.416  n=120
  1997s: mean=0.308  std=0.171  n=120
  2007s: mean=0.270  std=0.186  n=120
  2017s: mean=0.262  std=0.191  n=96


Saved: r2_timeseries.png


----

## Risk Price Visualization

The bar chart shows the annualized univariate FM risk price
$\hat{\lambda}_j^{\text{FM}} \times 12$ for all 41 characteristics,
coloured by family. Error bars are the Newey-West 95% confidence
interval $\hat{\lambda}_j \pm 1.96 \times \widehat{\text{SE}}_j^{\text{NW}}$.

Factors whose error bar does not cross the zero line are priced at
the 5% significance level. Note that the sign of $\hat{\lambda}_j$
reflects the direction of the raw characteristic in the CPZ dataset,
not the portfolio construction direction used in M1.

In [19]:
# Risk price and bar chart

print("Generating univariate risk price bar chart...")

fig = go.Figure()

for fam in CHAR_FAMILIES:
    chars   = CHAR_FAMILIES[fam]
    fam_res = fm_uni.loc[chars]
    lambdas = fam_res['Lambda (ann %)'].values
    errors  = (fam_res['SE (NW)'].values * np.sqrt(12) * 100 * 1.96)

    fig.add_trace(go.Bar(
        name         = fam,
        x            = chars,
        y            = lambdas,
        error_y      = dict(type='data', array=errors,
                            visible=True, thickness=1.5, width=4),
        marker_color = FAMILY_COLORS[fam],
        opacity      = 0.85,
        hovertemplate=(
            "<b>%{x}</b><br>"
            f"Family: {fam}<br>"
            "λ (ann%): %{y:.2f}%<extra></extra>"
        )
    ))

fig.add_hline(y=0,
              line=dict(color='black', width=1, dash='dash'),
              opacity=0.5)

fig.update_layout(
    title=dict(
        text=("<b>Fama-MacBeth Risk Price Estimates — Univariate</b><br>"
              "<sup>Annualized $\\hat{\\lambda}_j^{FM}$ | "
              "Error bars = NW 95% CI | 1967–2024</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Characteristic',
               tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(title='Risk Price λ (ann %)', ticksuffix='%',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    legend=dict(orientation='h', yanchor='bottom',
                y=1.02, xanchor='right', x=1),
    width=1100, height=520,
    margin=dict(l=60, r=60, t=100, b=130),
    hovermode='closest'
)
fig.show()
fig.write_image(os.path.join(FIG_DIR, "risk_prices_univariate.png"), scale=2)
print("Saved: risk_prices_univariate.png")

Generating univariate risk price bar chart...


Saved: risk_prices_univariate.png


In [20]:
print("=== Saving M2 outputs ===\n")

# Risk price tables
fm_uni.to_csv(   os.path.join(OUT_DIR, "lambda_univariate.csv"))
fm_ridge.to_csv( os.path.join(OUT_DIR, "lambda_ridge.csv"))
fm_lasso.to_csv( os.path.join(OUT_DIR, "lambda_lasso.csv"))
print("Saved: lambda_univariate.csv")
print("Saved: lambda_ridge.csv")
print("Saved: lambda_lasso.csv")

# Lambda time series — univariate feeds M4 signal evaluation
lambda_uni_ts.to_parquet(
    os.path.join(OUT_DIR, "lambda_timeseries_univariate.parquet"))
print("Saved: lambda_timeseries_univariate.parquet")

# R² time series
r2_ridge.to_frame('R2').to_csv(
    os.path.join(OUT_DIR, "r2_timeseries.csv"))
print("Saved: r2_timeseries.csv")

# File inventory
print(f"\nAll M2 output files:")
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    if os.path.isfile(full):
        print(f"  {f:<45} {os.path.getsize(full)/1024:>7.1f} KB")

# Summary
n_uni_196 = (fm_uni['t (NW)'].abs()   > 1.96).sum()
n_uni_300 = (fm_uni['t (NW)'].abs()   > 3.00).sum()
n_rdg_196 = (fm_ridge['t (NW)'].abs() > 1.96).sum()
n_lso_act = (fm_lasso['% Active']     > 50  ).sum()

print(f"""
{'='*60}
M2 COMPLETE
{'='*60}
Period:          1967-01 to 2024-12
Months (T):      {T}
Newey-West lag:  L = {L}

Primary result — Univariate FM:
  Priced |t_NW| > 1.96:  {n_uni_196}/41
  Priced |t_NW| > 3.00:  {n_uni_300}/41

Robustness:
  Ridge priced:          {n_rdg_196}/41
  LASSO active >50%:     {n_lso_act}/41

Cross-sectional R²:
  Ridge mean:  {r2_ridge.mean()*100:.2f}%
  LASSO mean:  {r2_lasso.mean()*100:.2f}%

Outputs:       {OUT_DIR}
Feeds into:    M3 — Factor Orthogonalization
               M4 — Signal Evaluation
{'='*60}
""")

=== Saving M2 outputs ===

Saved: lambda_univariate.csv
Saved: lambda_ridge.csv
Saved: lambda_lasso.csv
Saved: lambda_timeseries_univariate.parquet
Saved: r2_timeseries.csv

All M2 output files:
  lambda_lasso.csv                                  2.7 KB
  lambda_ridge.csv                                  2.5 KB
  lambda_timeseries_univariate.parquet            286.7 KB
  lambda_univariate.csv                             2.5 KB
  r2_timeseries.csv                                21.4 KB

M2 COMPLETE
Period:          1967-01 to 2024-12
Months (T):      696
Newey-West lag:  L = 5

Primary result — Univariate FM:
  Priced |t_NW| > 1.96:  28/41
  Priced |t_NW| > 3.00:  24/41

Robustness:
  Ridge priced:          35/41
  LASSO active >50%:     17/41

Cross-sectional R²:
  Ridge mean:  24.44%
  LASSO mean:  21.14%

Outputs:       C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS520_Machine Learning in Finance\Project\General ML4Trading Resource\Personal_Fundamental Factor Inv